In [ ]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 2.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                              COLETA DOS DADOS                                ║
# ║          Coleta, Validação e Auditoria dos Dados                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IDENTIFICAÇÃO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Autor       : Yago
#  Instituição : [Nome da Instituição]
#  Curso       : [Nome do Curso]
#  Data        : Março / 2026
#  Versão      : 3.0
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DESCRIÇÃO / FUNÇÃO DO SCRIPT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Este script implementa a etapa de coleta e auditoria de dados da arquitetura
#  de dados do TCC. Ele realiza o download automatizado, validação e catalogação
#  de 20 conjuntos de dados públicos relacionados ao setor elétrico brasileiro,
#  cobrindo o período de 2021 a 2026.
#
#  Os dados são provenientes de quatro fontes oficiais:
#    • CCEE  — Câmara de Comercialização de Energia Elétrica
#    • ONS   — Operador Nacional do Sistema Elétrico
#    • INMET — Instituto Nacional de Meteorologia
#    • BCB   — Banco Central do Brasil
#
#  Ao final da execução, o script gera:
#    (a) uma estrutura de diretórios organizada com todos os dados em formato CSV;
#    (b) tabelas de auditoria exibidas no terminal (via Rich);
#    (c) um relatório PDF de rastreabilidade da coleta.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ENTRADAS (INPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  O script não requer entradas manuais do usuário. Os parâmetros de controle
#  estão definidos nas constantes abaixo (seção "CONFIGURAÇÕES DO AMBIENTE"):
#
#    ANOS_ALVO   : lista de anos a serem coletados  (padrão: 2021–2026)
#    BASE_PATH   : caminho raiz de armazenamento
#                  → Google Drive (/content/drive/MyDrive/TCC_PLD_Project)
#                  → local (./TCC_DATA_MASTER) se Drive não estiver montado
#
#  Fontes de dados acessadas via HTTP/HTTPS (requerem conexão à internet):
#    • https://pda-download.ccee.org.br         (PLD horário)
#    • https://ons-aws-prod-opendata.s3.amazonaws.com  (múltiplos datasets ONS)
#    • https://portal.inmet.gov.br/uploads/dadoshistoricos  (ZIPs meteorológicos)
#    • https://apicarga.ons.org.br/prd/cargaverificada  (API REST ONS)
#    • https://olinda.bcb.gov.br/olinda/servico/PTAX  (API REST BCB)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SAÍDAS (OUTPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  1. Arquivos CSV  →  BASE_PATH / "2-DADOS" / "1-DADOS-BRUTOS" / <pasta_dataset>
#     Convenção de nomenclatura: <chave_dataset>_<ano>.csv  (ex.: pld_2023.csv)
#     Separador: ponto e vírgula (;) | Decimal: vírgula (,) | Encoding: UTF-8
#
#  2. Relatório PDF →  BASE_PATH / "8-RELATÓRIOS" / Relatorio_Coleta_<YYYYMMDD_HHMM>.pdf
#     Contém log completo de cada arquivo baixado (dataset, ano, status, auditoria).
#
#  3. Saída no terminal (Rich):
#     • Tabela de execução por dataset/ano com status (BAIXADO / MANTIDO / ERRO)
#     • Auditoria de integridade (sucessos × falhas)
#     • Relatório de datas finais por dataset
#     • Tabela 1 — Catálogo de datasets (origem, período, frequência, linhas, colunas)
#     • Tabela 2 — Status de atualização (última data capturada × data de referência)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DATASETS COLETADOS (resumo)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  N°  Chave               Origem  Frequência   Descrição
#  01  PLD                 CCEE    Horária      Preço de Liquidação das Diferenças
#  02  INTERCAMBIO         ONS     Horária      Intercâmbio Nacional entre subsistemas
#  03  BALANCO             ONS     Horária      Balanço Energético por subsistema
#  04  CMO                 ONS     Semi-horária Custo Marginal de Operação
#  05  CURVA               ONS     Horária      Curva de Carga do sistema
#  06  EAR                 ONS     Diária       Energia Armazenada nos reservatórios
#  07  ENA                 ONS     Diária       Energia Natural Afluente
#  08  CARGA               ONS     Diária       Carga de Energia por subsistema
#  09  CVU                 ONS     Semanal      Custo Variável Unitário (termelétricas)
#  10  VOLUME_ESPERA       ONS     Diária       Volume de Espera nos reservatórios
#  11  HIDRO               ONS     Horária      Dados Hidrológicos dos reservatórios
#  12  DISPONIBILIDADE     ONS     Horária      Disponibilidade das usinas
#  13  FATOR_CAPACIDADE    ONS     Diária       Fator de Capacidade das usinas
#  14  GERACAO             ONS     Horária      Geração por usina
#  15  VERTIDA             ONS     Horária      Energia Vertida e Turbinável
#  16  TERMICA             ONS     Horária      Geração Térmica de despacho
#  17  INMET               INMET   Horária      Variáveis meteorológicas (estações)
#  18  CARGA_VERIFICADA    ONS     Horária      Carga verificada via API REST
#  19  DOLAR               BCB     Diária       Cotação PTAX do dólar americano
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONSIDERAÇÕES INICIAIS E OBSERVAÇÕES TÉCNICAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  1. AMBIENTE DE EXECUÇÃO
#     O script foi desenvolvido e testado no Google Colab (Python 3.10+).
#     Pode ser executado localmente, mas requer ajuste manual de BASE_PATH.
#     Antes de executar, montar o Google Drive e instalar as dependências:
#
#         from google.colab import drive
#         drive.mount('/content/drive')
#
#         !pip install rich reportlab
#
#  2. POLÍTICA DE CACHE
#     Para evitar downloads redundantes, o script implementa cache local:
#     - Anos anteriores ao ano corrente: mantidos em disco se já existirem.
#     - Ano corrente: sempre re-baixados para garantir dados atualizados.
#     - INMET: dados históricos mantidos se o ZIP já foi extraído; ano corrente
#       é sempre pulado (o INMET não disponibiliza ZIP do ano em curso).
#
#  3. SEGURANÇA E CONEXÃO
#     Certificados SSL são desabilitados (verify=False) para compatibilidade com
#     endpoints da CCEE que utilizam certificados auto-assinados. Isso é aceitável
#     em ambiente acadêmico/pesquisa, mas NÃO deve ser usado em produção.
#     A sessão HTTP implementa retry automático (5 tentativas, backoff exponencial)
#     para mitigar falhas transitórias de rede.
#
#  4. TIMEZONES
#     Alguns endpoints da ONS retornam timestamps com fuso horário (tz-aware).
#     O script normaliza todos os timestamps para tz-naive (UTC sem fuso) antes
#     de comparações, evitando TypeError entre objetos tz-aware e tz-naive.
#
#  5. LEITURA FLEXÍVEL DE CSV
#     A função _read_csv_flex() tenta automaticamente separadores (;  e ,) e
#     encodings (latin-1 e UTF-8), pois diferentes endpoints da ONS adotam
#     convenções distintas. Linhas mal formadas são ignoradas (on_bad_lines="skip").
#
#  6. DADOS DO ANO CORRENTE (2026)
#     Arquivos do ano em curso podem estar incompletos. O campo de auditoria
#     sinaliza "(Parcial)" automaticamente sempre que ano == ANO_ATUAL.
#
#  7. LIMITES CONHECIDOS
#     - Commodities (petróleo, gás) via Yahoo Finance foram removidos desta versão
#       por instabilidade da API. Podem ser reintegrados em versão futura.
#     - O endpoint da CCEE para PLD utiliza URLs temporárias (content tokens)
#       que podem expirar; atualizar manualmente em DATASETS_CONFIG se necessário.
#     - Dados do INMET para o ano corrente não estão disponíveis em formato ZIP;
#       coleta alternativa via API do INMET pode ser implementada futuramente.
#
#  8. REPRODUTIBILIDADE
#     Todos os arquivos gerados incluem o timestamp de execução no nome (PDF)
#     ou no caminho (pasta por ano, para INMET), garantindo rastreabilidade total.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DEPENDÊNCIAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Biblioteca        Versão mínima  Finalidade
#  numpy             1.23           Operações numéricas auxiliares
#  pandas            1.5            Manipulação de DataFrames e datas
#  requests          2.28           Download HTTP com sessão e retry
#  urllib3           1.26           Supressão de avisos SSL
#  rich              13.0           Tabelas e painéis coloridos no terminal
#  reportlab         3.6            Geração do relatório em PDF
#
#  Instalação rápida (Google Colab / pip):
#      !pip install rich reportlab
#  (numpy, pandas, requests e urllib3 já estão disponíveis no Colab por padrão)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ESTRUTURA DE DIRETÓRIOS GERADA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  TCC_PLD_Project/
#  ├── 2-DADOS/
#  │   └── 1-DADOS-BRUTOS/
#  │       ├── 1-CCEE-PLD-HORARIO/         pld_2021.csv … pld_2026.csv
#  │       ├── 2-ONS-BALANÇO-DE-ENERGIA-HORARIO/
#  │       ├── 3-ONS-HIDRAULICOS-RESERVATORIOS-HORARIO/
#  │       ├── 4-ONS-GERACAO-USINA-HORARIO/
#  │       ├── 5-ONS-CMO-HORARIO/
#  │       ├── 6-ONS-ENERGIA-VERTIDA-TURBINAVEL/
#  │       ├── 8-ONS-CURVA-CARGA/
#  │       ├── 9-ONS-DISPONIBILIDADE-USINA/
#  │       ├── 10-ONS-GERACAO-TERMICA-DESPACHO/
#  │       ├── 11-INMET-METEOROLOGICOS/
#  │       │   ├── 2021/   (ZIPs extraídos por ano)
#  │       │   └── …
#  │       ├── 12-ONS-EAR/
#  │       ├── 13-ONS-ENA/
#  │       ├── 14-ONS-CARGA/
#  │       ├── 15-ONS-CVU/
#  │       ├── 16-ONS-CARGA-VERIFICADA/
#  │       ├── 17-ONS-INTERCAMBIO/
#  │       ├── 17-ONS-VOLUME-ESPERA/
#  │       ├── 18-ONS-FATOR-CAPACIDADE/
#  │       └── 19-BCB-DOLAR/
#  └── 8-RELATÓRIOS/
#      └── Relatorio_Coleta_<YYYYMMDD_HHMM>.pdf
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  COMO EXECUTAR (Google Colab)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Célula 1 — Montar o Drive:
#      from google.colab import drive
#      drive.mount('/content/drive')
#
#  Célula 2 — Instalar dependências:
#      !pip install rich reportlab
#
#  Célula 3 — Executar o script:
#      exec(open('arquitetura_dados_tcc_v10.py').read())
#   ou simplesmente executar este arquivo como módulo principal.
#
# ══════════════════════════════════════════════════════════════════════════════

import logging
import time
import warnings
import zipfile
import os
from datetime import datetime as dt
from io import BytesIO
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

# --- Bibliotecas de Dados & Rede ---
import numpy as np
import pandas as pd
import requests
import urllib3
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- Bibliotecas Visuais (Rich) ---
try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn
    from rich import box
    from rich.theme import Theme
    from rich.text import Text
    from rich.rule import Rule
except ImportError:
    print("❌ Erro: Instale as libs: pip install rich reportlab")
    exit()

# --- Bibliotecas PDF ---
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table as PDFTable, TableStyle

# ==============================================================================
# ⚙️ CONFIGURAÇÕES DO AMBIENTE
# ==============================================================================
custom_theme = Theme({
    "info":    "dim cyan",
    "warning": "bold yellow",
    "error":   "bold red",
    "success": "bold green",
    "header":  "bold white on blue",
})
console = Console(theme=custom_theme)

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

ANOS_ALVO = [2021, 2022, 2023, 2024, 2025, 2026]
ANO_ATUAL = dt.now().year
DATA_REF   = dt.now()   # Timestamp fixo no início da execução

BASE_PATH = Path("/content/drive/MyDrive/TCC_PLD_Project")
if not BASE_PATH.parent.exists():
    BASE_PATH = Path("./TCC_DATA_MASTER")

BASE_PATH.mkdir(parents=True, exist_ok=True)
RAW_DATA_PATH = BASE_PATH / "2-DADOS" / "1-DADOS-BRUTOS"
REPORT_PATH   = BASE_PATH / "8-RELATÓRIOS"

# ==============================================================================
# 📋 CATÁLOGO DE DADOS  (sem Commodities/Yahoo)
# ==============================================================================
DATASETS_CONFIG = [
    {
        "key": "PLD", "name": "Preço (PLD) CCEE",
        "folder": "1-CCEE-PLD-HORARIO",
        "origem": "CCEE", "freq": "Horária", "periodo": "2021–2026",
        "type": "simple_dict", "engine": "simple",
        "urls": {
            2021: "https://pda-download.ccee.org.br/SMpDR_R7SCOOj6pMbk1BJg/content",
            2022: "https://pda-download.ccee.org.br/0YTnGY1jRb-tarnKnSNT9g/content",
            2023: "https://pda-download.ccee.org.br/HH4Xegm7R56M_H4qPNOvaw/content",
            2024: "https://pda-download.ccee.org.br/rMsBwN6TT-WUW2_LbGUvkw/content",
            2025: "https://pda-download.ccee.org.br/korJMXwpSLGyVlpRMQWduA/content",
            2026: "https://pda-download.ccee.org.br/6A5wq97KTCWv_bvs3CqsQQ/content",
        },
    },
    {
        "key": "INTERCAMBIO", "name": "Intercâmbio Nacional",
        "folder": "17-ONS-INTERCAMBIO",
        "origem": "ONS", "freq": "Horária", "periodo": "2021–2026",
        "type": "simple_dict", "engine": "simple",
        "urls": {
            2021: "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/intercambio_nacional_ho/INTERCAMBIO_NACIONAL_2021.csv",
            2022: "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/intercambio_nacional_ho/INTERCAMBIO_NACIONAL_2022.csv",
            2023: "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/intercambio_nacional_ho/INTERCAMBIO_NACIONAL_2023.csv",
            2024: "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/intercambio_nacional_ho/INTERCAMBIO_NACIONAL_2024.csv",
            2025: "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/intercambio_nacional_ho/INTERCAMBIO_NACIONAL_2025.csv",
            2026: "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/intercambio_nacional_ho/INTERCAMBIO_NACIONAL_2026.csv",
        },
    },
    {
        "key": "BALANCO", "name": "Balanço Energético",
        "folder": "2-ONS-BALANÇO-DE-ENERGIA-HORARIO",
        "origem": "ONS", "freq": "Horária", "periodo": "2021–2026",
        "type": "simple_template", "engine": "simple",
        "url_template": "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/balanco_energia_subsistema_ho/BALANCO_ENERGIA_SUBSISTEMA_{ano}.csv",
    },
    {
        "key": "CMO", "name": "Custo Marginal (CMO)",
        "folder": "5-ONS-CMO-HORARIO",
        "origem": "ONS", "freq": "Semihorária", "periodo": "2021–2026",
        "type": "simple_template", "engine": "simple",
        "url_template": "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/cmo_tm/CMO_SEMIHORARIO_{ano}.csv",
    },
    {
        "key": "CURVA", "name": "Curva de Carga",
        "folder": "8-ONS-CURVA-CARGA",
        "origem": "ONS", "freq": "Horária", "periodo": "2021–2026",
        "type": "simple_template", "engine": "simple",
        "url_template": "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/curva_carga_ho/CURVA_CARGA_{ano}.csv",
    },
    {
        "key": "EAR", "name": "Energia Armazenada (EAR)",
        "folder": "12-ONS-EAR",
        "origem": "ONS", "freq": "Diária", "periodo": "2021–2026",
        "type": "simple_template", "engine": "simple",
        "url_template": "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/ear_subsistema_di/EAR_DIARIO_SUBSISTEMA_{ano}.csv",
    },
    {
        "key": "ENA", "name": "Energia Natural (ENA)",
        "folder": "13-ONS-ENA",
        "origem": "ONS", "freq": "Diária", "periodo": "2021–2026",
        "type": "simple_template", "engine": "simple",
        "url_template": "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/ena_subsistema_di/ENA_DIARIO_SUBSISTEMA_{ano}.csv",
    },
    {
        "key": "CARGA", "name": "Carga de Energia",
        "folder": "14-ONS-CARGA",
        "origem": "ONS", "freq": "Diária", "periodo": "2021–2026",
        "type": "simple_template", "engine": "simple",
        "url_template": "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_{ano}.csv",
    },
    {
        "key": "CVU", "name": "Custo Variável (CVU)",
        "folder": "15-ONS-CVU",
        "origem": "ONS", "freq": "Semanal", "periodo": "2021–2026",
        "type": "simple_template", "engine": "simple",
        "url_template": "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/cvu_usitermica_se/CVU_USINA_TERMICA_{ano}.csv",
    },
    {
        "key": "VOLUME_ESPERA", "name": "Volume de Espera",
        "folder": "17-ONS-VOLUME-ESPERA",
        "origem": "ONS", "freq": "Diária", "periodo": "2021–2026",
        "type": "simple_template", "engine": "simple",
        "url_template": "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/res_volumeespera_di/RES_VOLUMEESPERA_{ano}.csv",
    },
    {
        "key": "HIDRO", "name": "Dados Hidrológicos",
        "folder": "3-ONS-HIDRAULICOS-RESERVATORIOS-HORARIO",
        "origem": "ONS", "freq": "Horária", "periodo": "2021–2026",
        "type": "monthly", "engine": "simple",
        "url_template": "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/dados_hidrologicos_ho/DADOS_HIDROLOGICOS_HO_{ano}_{mes}.csv",
    },
    {
        "key": "DISPONIBILIDADE", "name": "Disponibilidade Usina",
        "folder": "9-ONS-DISPONIBILIDADE-USINA",
        "origem": "ONS", "freq": "Horária", "periodo": "2021–2026",
        "type": "monthly", "engine": "simple",
        "url_template": "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/disponibilidade_usina_ho/DISPONIBILIDADE_USINA_{ano}_{mes}.csv",
    },
    {
        "key": "FATOR_CAPACIDADE", "name": "Fator Capacidade",
        "folder": "18-ONS-FATOR-CAPACIDADE",
        "origem": "ONS", "freq": "Diária", "periodo": "2021–2026",
        "type": "monthly", "engine": "simple",
        "url_template": "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/fator_capacidade_2_di/FATOR_CAPACIDADE-2_{ano}_{mes}.csv",
    },
    {
        "key": "GERACAO", "name": "Geração Usinas",
        "folder": "4-ONS-GERACAO-USINA-HORARIO",
        "origem": "ONS", "freq": "Horária", "periodo": "2021–2026",
        "type": "custom",
    },
    {
        "key": "VERTIDA", "name": "Energia Vertida",
        "folder": "6-ONS-ENERGIA-VERTIDA-TURBINAVEL",
        "origem": "ONS", "freq": "Horária", "periodo": "2021–2026",
        "type": "custom",
    },
    {
        "key": "TERMICA", "name": "Geração Térmica",
        "folder": "10-ONS-GERACAO-TERMICA-DESPACHO",
        "origem": "ONS", "freq": "Horária", "periodo": "2021–2026",
        "type": "custom",
    },
    {
        "key": "INMET", "name": "Meteorologia (INMET)",
        "folder": "11-INMET-METEOROLOGICOS",
        "origem": "INMET", "freq": "Horária", "periodo": "2021–2025",
        "type": "custom",
    },
    {
        "key": "CARGA_VERIFICADA", "name": "Carga Verificada (API)",
        "folder": "16-ONS-CARGA-VERIFICADA",
        "origem": "ONS", "freq": "Horária", "periodo": "2021–2026",
        "type": "custom",
    },
    {
        "key": "DOLAR", "name": "Cotação Dólar",
        "folder": "19-BCB-DOLAR",
        "origem": "BCB", "freq": "Diária", "periodo": "2021–2026",
        "type": "custom",
    },
    # ── Commodities (Yahoo) REMOVIDO ──
]

# Regras especiais de coluna de data por dataset
DATE_RULES: Dict[str, str] = {
    "CVU":              "dat_iniciosemana",
    "Custo Variável (CVU)": "dat_iniciosemana",
}

# Mapeamento de colunas de data conhecidas por nome de dataset
DATE_COLUMN_MAP: Dict[str, List[str]] = {
    "Preço (PLD) CCEE":        ["mes_referencia", "din_instante", "data"],
    "Energia Armazenada (EAR)": ["din_instante", "dat_referencia", "data"],
    "Energia Natural (ENA)":    ["din_instante", "dat_referencia", "data"],
    "Carga de Energia":         ["din_instante", "dat_referencia", "data"],
    "Custo Variável (CVU)":     ["dat_iniciosemana", "data"],
    "Meteorologia (INMET)":     ["data", "hora_utc", "datetime"],
    "Carga Verificada (API)":   ["din_instante", "datahora", "data"],
    "Cotação Dólar":            ["datahoracotacao", "datahora", "data"],
}

# ==============================================================================
# ⏱️ UTILITÁRIO DE TEMPO
# ==============================================================================
class TimeTracker:
    def __init__(self):
        self.start    = 0
        self.duration = 0

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.duration = time.perf_counter() - self.start

    @property
    def formatted(self) -> str:
        if self.duration < 1:
            return f"{self.duration * 1000:.0f}ms"
        return f"{self.duration:.2f}s" if self.duration < 60 else f"{self.duration / 60:.1f}m"

# ==============================================================================
# 🔧 UTILITÁRIOS DE LEITURA & PADRONIZAÇÃO
# ==============================================================================
def _read_csv_flex(path: Union[str, Path], nrows: Optional[int] = None) -> pd.DataFrame:
    """Lê CSV tentando múltiplos separadores e encodings. Suprime DtypeWarning."""
    for sep in [";", ","]:
        for enc in ["latin1", "utf-8"]:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    return pd.read_csv(
                        path, sep=sep, encoding=enc,
                        on_bad_lines="skip", nrows=nrows,
                        low_memory=False,
                    )
            except Exception:
                continue
    raise Exception(f"Não foi possível ler: {path}")


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Padroniza nomes de colunas para snake_case sem acentos."""
    replacements = str.maketrans("çãâéêíóôú", "caaeeiooú")
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .map(lambda c: c.translate(replacements))
    )
    return df


def _find_date_column(df: pd.DataFrame, hints: Optional[List[str]] = None) -> Optional[str]:
    """
    Detecta a coluna de data.
    1. Tenta as hints fornecidas (lista de nomes candidatos).
    2. Varre por keywords genéricas.
    Valida se ao menos 50% dos valores fazem parse como datetime.
    """
    candidates = list(hints or []) + [
        "data", "date", "instante", "hora", "referencia",
        "dat_", "inicio", "fim", "din_", "datahora",
    ]
    for hint in (hints or []):
        if hint in df.columns:
            parsed = pd.to_datetime(df[hint], errors="coerce")
            if parsed.notna().sum() > len(df) * 0.3:
                return hint

    for col in df.columns:
        col_lower = col.lower()
        if any(k in col_lower for k in candidates):
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().sum() > len(df) * 0.3:
                return col
    return None


def _strip_tz(ts) -> Optional[pd.Timestamp]:
    """
    Normaliza qualquer Timestamp para tz-naive (UTC → local drop).
    Evita TypeError ao comparar tz-naive vs tz-aware.
    """
    if ts is None or (isinstance(ts, float) and np.isnan(ts)):
        return None
    try:
        ts = pd.Timestamp(ts)
        if ts is pd.NaT:
            return None
        if ts.tzinfo is not None:
            ts = ts.tz_localize(None)
        return ts
    except Exception:
        return None


def _classify_status(last_date) -> Tuple[str, str]:
    """
    Retorna (label, cor_rich) com base na defasagem em relação a hoje.
    → verde  : atualizado (≤ 1 dia)
    → amarelo: atenção    (2–7 dias)
    → vermelho: defasado  (> 7 dias)
    """
    ts = _strip_tz(last_date)
    if ts is None:
        return "Sem dado", "dim"
    try:
        diff = (pd.Timestamp.now() - ts).days
    except Exception:
        return "Erro", "bold red"

    if diff <= 1:
        return "✔ Atualizado", "bold green"
    elif diff <= 7:
        return "⚠ Atenção",    "bold yellow"
    else:
        return "✖ Defasado",   "bold red"

# ==============================================================================
# 🏗️ ENGINE DE COLETA
# ==============================================================================
class SecureDataCollector:
    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.session    = requests.Session()
        retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
        self.session.mount("https://", HTTPAdapter(max_retries=retries))
        self.session.headers.update({"User-Agent": "Mozilla/5.0"})
        self.session.verify = False
        self.logs: List[List[str]] = []

    # --------------------------------------------------------------------------
    # 📝 LOGGING
    # --------------------------------------------------------------------------
    def log_event(self, dataset: str, ano, status: str, msg: str, tempo: str = "-") -> None:
        color_map = {
            "BAIXADO":  "green",
            "MANTIDO":  "blue",
            "ERRO":     "red",
            "PULADO":   "white",
            "EXTRAÍDO": "magenta",
            "ATUALIZADO": "cyan",
        }
        color = color_map.get(status, "white")
        if status != "MANTIDO":
            console.print(f"   │ {str(ano):<6} │ [{color}]{status:<10}[/] │ {msg} ({tempo})")
        self.logs.append([dataset, str(ano), status, msg, tempo])

    # --------------------------------------------------------------------------
    # 📥 DOWNLOADS
    # --------------------------------------------------------------------------
    def _download_simple(self, url: str) -> Optional[BytesIO]:
        try:
            r = requests.get(url, timeout=120, verify=False)
            r.raise_for_status()
            return BytesIO(r.content)
        except Exception:
            return None

    def _download_secure(self, url: str) -> Optional[BytesIO]:
        try:
            with self.session.get(url, stream=True, timeout=120, verify=False) as r:
                r.raise_for_status()
                return BytesIO(r.content)
        except Exception:
            return None

    # --------------------------------------------------------------------------
    # 🔬 AUDITORIA DE DADOS
    # --------------------------------------------------------------------------
    def _audit_data(self, df: pd.DataFrame, ano: int, key: str) -> str:
        """Valida shape, memória e preenchimento; retorna string de resumo."""
        if df.empty:
            return "ERRO: Vazio"
        size_mb   = df.memory_usage(deep=True).sum() / (1024 * 1024)
        fill_rate = (df.count().sum() / df.size) * 100 if df.size > 0 else 0
        suffix    = " (Parcial)" if int(ano) == ANO_ATUAL else ""
        return f"{len(df)} lins | {size_mb:.1f}MB | {fill_rate:.0f}% OK{suffix}"

    def _get_total_rows(self, folder: Path) -> Tuple[int, int]:
        """Soma linhas de TODOS os CSVs da pasta (período completo 2021–2026)."""
        csv_files = sorted(folder.glob("*.csv"))
        if not csv_files:
            return 0, 0
        total_rows = 0
        n_cols     = 0
        for i, f in enumerate(csv_files):
            try:
                df = _read_csv_flex(f, nrows=None)
                total_rows += len(df)
                if i == 0:
                    n_cols = len(df.columns)
            except Exception:
                pass
        return total_rows, n_cols

    def _get_last_date(self, name: str, folder: Path) -> Optional[pd.Timestamp]:
        """Varre todos os CSVs da pasta e retorna a data máxima global."""
        csv_files = sorted(folder.glob("*.csv"), reverse=True)
        if not csv_files:
            return None

        max_date: Optional[pd.Timestamp] = None

        for csv_path in csv_files:
            try:
                df = _read_csv_flex(csv_path)
                df = _normalize_columns(df)

                date_ts: Optional[pd.Timestamp] = None

                if name == "Preço (PLD) CCEE":
                    needed = {"mes_referencia", "dia", "hora"}
                    if needed.issubset(df.columns):
                        df["_dt"] = pd.to_datetime(
                            df["mes_referencia"].astype(str)
                            + df["dia"].astype(str).str.zfill(2)
                            + df["hora"].astype(str).str.zfill(2),
                            format="%Y%m%d%H", errors="coerce",
                        )
                        date_ts = _strip_tz(df["_dt"].max())

                if date_ts is None:
                    hints = DATE_COLUMN_MAP.get(name, [])
                    date_col = _find_date_column(df, hints)
                    if date_col:
                        date_ts = _strip_tz(
                            pd.to_datetime(df[date_col], errors="coerce").max()
                        )

                if date_ts is not None:
                    if max_date is None or date_ts > max_date:
                        max_date = date_ts

            except Exception:
                continue

        return max_date

    # --------------------------------------------------------------------------
    # 💾 PERSISTÊNCIA
    # --------------------------------------------------------------------------
    def save_df(self, df: pd.DataFrame, path: Path) -> None:
        df.to_csv(path, index=False, sep=";", decimal=",")

    # --------------------------------------------------------------------------
    # ⚙️ PROCESSADORES GENÉRICOS
    # --------------------------------------------------------------------------
    def process_generic(self, config: dict) -> None:
        folder = self.output_dir / config["folder"]
        folder.mkdir(parents=True, exist_ok=True)
        console.print(f"\n[bold blue]➤ {config['name']}[/]")
        for ano in ANOS_ALVO:
            path = folder / f"{config['key'].lower()}_{ano}.csv"
            if path.exists() and int(ano) < ANO_ATUAL:
                self.log_event(config["name"], ano, "MANTIDO", "Cache Local")
                continue
            url = (
                config["urls"].get(ano)
                if config["type"] == "simple_dict"
                else config["url_template"].format(ano=ano)
            )
            if not url:
                continue
            with TimeTracker() as t:
                content = (
                    self._download_simple(url)
                    if config.get("engine") == "simple"
                    else self._download_secure(url)
                )
                if content:
                    try:
                        df = pd.read_csv(content, sep=";", encoding="utf-8", low_memory=False)
                    except Exception:
                        content.seek(0)
                        df = pd.read_csv(content, sep=",", encoding="utf-8", low_memory=False)
                    audit = self._audit_data(df, ano, config["key"])
                    status = "ATUALIZADO" if int(ano) == ANO_ATUAL and path.exists() else "BAIXADO"
                    self.save_df(df, path)
                    self.log_event(config["name"], ano, status, audit, t.formatted)
                else:
                    self.log_event(config["name"], ano, "ERRO", "Falha HTTP", t.formatted)

    def process_monthly(self, config: dict) -> None:
        folder = self.output_dir / config["folder"]
        folder.mkdir(parents=True, exist_ok=True)
        console.print(f"\n[bold blue]➤ {config['name']} (Mensal)[/]")
        for ano in ANOS_ALVO:
            path = folder / f"{config['key'].lower()}_{ano}.csv"
            if path.exists() and int(ano) < ANO_ATUAL:
                self.log_event(config["name"], ano, "MANTIDO", "Cache Local")
                continue
            with TimeTracker() as t:
                dfs = []
                for mes in range(1, 13):
                    url = config["url_template"].format(ano=ano, mes=f"{mes:02d}")
                    content = (
                        self._download_simple(url)
                        if config.get("engine") == "simple"
                        else self._download_secure(url)
                    )
                    if content:
                        try:
                            dfs.append(pd.read_csv(content, sep=";", low_memory=False))
                        except Exception:
                            pass
                if dfs:
                    full_df = pd.concat(dfs, ignore_index=True)
                    audit   = self._audit_data(full_df, ano, config["key"])
                    status  = "ATUALIZADO" if int(ano) == ANO_ATUAL and path.exists() else "BAIXADO"
                    self.save_df(full_df, path)
                    self.log_event(config["name"], ano, status, audit, t.formatted)
                else:
                    self.log_event(config["name"], ano, "ERRO", "Vazio", t.formatted)

    # --------------------------------------------------------------------------
    # ⚙️ PROCESSADORES CUSTOM
    # --------------------------------------------------------------------------
    def process_custom_inmet(self, folder: Path) -> None:
        console.print(f"\n[bold blue]➤ Meteorologia INMET[/]")
        folder.mkdir(parents=True, exist_ok=True)

        for ano in ANOS_ALVO:
            if int(ano) == ANO_ATUAL:
                self.log_event("INMET", ano, "PULADO", "Ano corrente (s/ ZIP no INMET)")
                continue

            ano_folder = folder / str(ano)
            csvs_extraidos = list(ano_folder.glob("**/*.CSV")) + list(ano_folder.glob("**/*.csv"))

            if csvs_extraidos:
                self.log_event("INMET", ano, "MANTIDO", f"Cache Local ({len(csvs_extraidos)} CSVs)")
                continue

            url = f"https://portal.inmet.gov.br/uploads/dadoshistoricos/{ano}.zip"
            with TimeTracker() as t:
                content = self._download_secure(url)
                if content:
                    try:
                        with zipfile.ZipFile(content) as z:
                            z.extractall(folder / str(ano))
                        n_extraidos = len(list((folder / str(ano)).glob("**/*.CSV")))
                        self.log_event("INMET", ano, "EXTRAÍDO",
                                       f"✅ ZIP extraído ({n_extraidos} arquivos)", t.formatted)
                    except Exception as e:
                        self.log_event("INMET", ano, "ERRO", f"Erro ZIP: {e}", t.formatted)
                else:
                    self.log_event("INMET", ano, "ERRO", "Falha Download", t.formatted)

    def process_custom_geracao(self, folder: Path) -> None:
        console.print(f"\n[bold blue]➤ Geração Usinas[/]")
        folder.mkdir(parents=True, exist_ok=True)
        for ano in ANOS_ALVO:
            path = folder / f"geracao_usina_{ano}.csv"
            if path.exists() and int(ano) < ANO_ATUAL:
                self.log_event("Geração", ano, "MANTIDO", "Cache Local")
                continue
            with TimeTracker() as t:
                dfs   = []
                meses = [0] if ano == 2021 else range(1, 13)
                for m in meses:
                    url = (
                        f"https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/geracao_usina_2_ho/"
                        f"GERACAO_USINA-2_{ano}{'' if m == 0 else '_' + f'{m:02d}'}.csv"
                    )
                    c = self._download_simple(url)
                    if c:
                        try:
                            dfs.append(pd.read_csv(c, sep=";", low_memory=False))
                        except Exception:
                            pass
                if dfs:
                    df = pd.concat(dfs)
                    status = "ATUALIZADO" if int(ano) == ANO_ATUAL and path.exists() else "BAIXADO"
                    self.save_df(df, path)
                    self.log_event("Geração", ano, status, self._audit_data(df, ano, "GER"), t.formatted)
                else:
                    self.log_event("Geração", ano, "ERRO", "Vazio/Falha", t.formatted)

    def process_custom_vertida(self, folder: Path) -> None:
        console.print(f"\n[bold blue]➤ Energia Vertida[/]")
        folder.mkdir(parents=True, exist_ok=True)
        for ano in ANOS_ALVO:
            path = folder / f"energia_vertida_{ano}.csv"
            if path.exists() and int(ano) < ANO_ATUAL:
                self.log_event("Vertida", ano, "MANTIDO", "Cache Local")
                continue
            with TimeTracker() as t:
                dfs   = []
                meses = [0] if ano <= 2023 else range(1, 13)
                for m in meses:
                    url = (
                        f"https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/energia_vertida_turbinavel_ho/"
                        f"ENERGIA_VERTIDA_TURBINAVEL_{ano}{'' if m == 0 else '_' + f'{m:02d}'}.csv"
                    )
                    c = self._download_simple(url)
                    if c:
                        try:
                            dfs.append(pd.read_csv(c, sep=";", low_memory=False))
                        except Exception:
                            pass
                if dfs:
                    df = pd.concat(dfs)
                    status = "ATUALIZADO" if int(ano) == ANO_ATUAL and path.exists() else "BAIXADO"
                    self.save_df(df, path)
                    self.log_event("Vertida", ano, status, self._audit_data(df, ano, "VER"), t.formatted)
                else:
                    self.log_event("Vertida", ano, "ERRO", "Vazio/Falha", t.formatted)

    def process_custom_termica(self, folder: Path) -> None:
        console.print(f"\n[bold blue]➤ Geração Térmica[/]")
        folder.mkdir(parents=True, exist_ok=True)
        for ano in ANOS_ALVO:
            path = folder / f"geracao_termica_{ano}.csv"
            if path.exists() and int(ano) < ANO_ATUAL:
                self.log_event("Térmica", ano, "MANTIDO", "Cache Local")
                continue
            with TimeTracker() as t:
                dfs   = []
                meses = [0] if ano == 2021 else range(1, 13)
                for m in meses:
                    url = (
                        f"https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/geracao_termica_despacho_2_ho/"
                        f"GERACAO_TERMICA_DESPACHO{'' if m == 0 else '-2'}_{ano}"
                        f"{'' if m == 0 else '_' + f'{m:02d}'}.csv"
                    )
                    c = self._download_simple(url)
                    if c:
                        try:
                            dfs.append(pd.read_csv(c, sep=";", low_memory=False))
                        except Exception:
                            pass
                if dfs:
                    df = pd.concat(dfs)
                    status = "ATUALIZADO" if int(ano) == ANO_ATUAL and path.exists() else "BAIXADO"
                    self.save_df(df, path)
                    self.log_event("Térmica", ano, status, self._audit_data(df, ano, "TER"), t.formatted)
                else:
                    self.log_event("Térmica", ano, "ERRO", "Vazio/Falha", t.formatted)

    def process_custom_carga_verificada(self, folder: Path) -> None:
        console.print(f"\n[bold blue]➤ Carga Verificada (API)[/]")
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / "carga_verificada_2021_2026.csv"
        with TimeTracker() as t:
            url   = "https://apicarga.ons.org.br/prd/cargaverificada"
            dados = []
            for sub in ["SECO", "S", "NE", "N"]:
                try:
                    r = self.session.get(
                        url,
                        params={"dat_inicio": "2021-01-01",
                                "dat_fim": f"{ANO_ATUAL}-12-31",
                                "cod_areacarga": sub},
                        timeout=30,
                    )
                    if r.ok:
                        dados.extend(r.json())
                except Exception:
                    pass
            if dados:
                df = pd.DataFrame(dados)
                self.save_df(df, path)
                status = "ATUALIZADO" if path.exists() else "BAIXADO"
                self.log_event("Carga Verif.", "21-26", status,
                               self._audit_data(df, ANO_ATUAL, "API"), t.formatted)
            else:
                self.log_event("Carga Verif.", "21-26", "ERRO", "Falha API", t.formatted)

    def process_custom_dolar(self, folder: Path) -> None:
        console.print(f"\n[bold blue]➤ Dólar PTAX[/]")
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / "dolar_ptax_2021_2026.csv"
        url = (
            f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
            f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
            f"?@dataInicial='01-01-2021'&@dataFinalCotacao='12-31-{ANO_ATUAL}'"
            f"&$top=100000&$format=json"
        )
        with TimeTracker() as t:
            try:
                r = self.session.get(url, timeout=30)
                if r.ok:
                    df = pd.DataFrame(r.json()["value"])
                    self.save_df(df, path)
                    status = "ATUALIZADO" if path.exists() else "BAIXADO"
                    self.log_event("Dólar", "21-26", status,
                                   self._audit_data(df, ANO_ATUAL, "USD"), t.formatted)
                else:
                    self.log_event("Dólar", "21-26", "ERRO", f"HTTP {r.status_code}", t.formatted)
            except Exception as e:
                self.log_event("Dólar", "21-26", "ERRO", str(e)[:60], t.formatted)

    # --------------------------------------------------------------------------
    # 🔍 AUDITORIA DE INTEGRIDADE
    # --------------------------------------------------------------------------
    def auditar_downloads(self) -> None:
        console.print("\n[bold white on blue] 🔎 AUDITORIA DE INTEGRIDADE [/]")
        ok   = sum(1 for l in self.logs if l[2] in ["BAIXADO", "MANTIDO", "EXTRAÍDO", "ATUALIZADO"])
        erro = sum(1 for l in self.logs if l[2] == "ERRO")
        for l in self.logs:
            if l[2] == "ERRO":
                console.print(f"[red] ❌ Falha: {l[0]} ({l[1]}) - {l[3]}[/]")
        console.print(f"\n[bold green]✅ Sucessos: {ok}[/] | [bold red]⚠️ Falhas: {erro}[/]")

    # --------------------------------------------------------------------------
    # 📅 RELATÓRIO DATAS FINAIS
    # --------------------------------------------------------------------------
    def relatorio_datas_finais(self) -> None:
        console.print("\n[bold white on blue] 📅 RELATÓRIO DE DATAS FINAIS (LATEST DATA) [/]")
        t = Table(box=box.ROUNDED)
        t.add_column("Dataset", style="cyan")
        t.add_column("Última Data Capturada", style="bold green", justify="center")

        for config in DATASETS_CONFIG:
            folder = self.output_dir / config["folder"]
            max_d  = self._get_last_date(config["name"], folder)

            if max_d is not None and pd.notnull(max_d):
                t.add_row(config["name"], max_d.strftime("%d/%m/%Y"))
            else:
                t.add_row(config["name"], "[dim]N/A (S/ Coluna Temporal)[/]")

        console.print(t)

    # --------------------------------------------------------------------------
    # 📊 TABELA 1 — CATÁLOGO DE DATASETS
    # --------------------------------------------------------------------------
    def _build_catalog_table(self) -> None:
        console.print()
        console.print(Panel.fit(
            "[bold white]📋 TABELA 1 — CATÁLOGO DE DATASETS[/]",
            style="bold blue",
        ))

        tbl = Table(box=box.ROUNDED, show_lines=False, header_style="bold cyan")
        tbl.add_column("N°",         style="dim",        width=4,  justify="right")
        tbl.add_column("Origem",     style="cyan",       width=8)
        tbl.add_column("Dataset",    style="bold white", width=26)
        tbl.add_column("Período",    width=12,           justify="center")
        tbl.add_column("Frequência", width=13,           justify="center")
        tbl.add_column("Colunas",    width=8,            justify="right")
        tbl.add_column("Linhas",     width=14,           justify="right")
        tbl.add_column("Status",     width=14,           justify="center")

        for i, config in enumerate(DATASETS_CONFIG, start=1):
            folder = self.output_dir / config["folder"]

            if config["key"] == "INMET":
                all_csvs = list(folder.glob("**/*.csv")) + list(folder.glob("**/*.CSV"))
                if all_csvs:
                    tbl.add_row(
                        f"{i:02d}", config.get("origem", "—"), config["name"],
                        config.get("periodo", "2021–2026"), config.get("freq", "—"),
                        "N/A", f"{len(all_csvs)} arq.", "[dim]ZIP/Pasta[/]",
                    )
                else:
                    tbl.add_row(
                        f"{i:02d}", config.get("origem", "—"), config["name"],
                        config.get("periodo", "2021–2026"), config.get("freq", "—"),
                        "N/A", "N/A", "[bold red]✖ Ausente[/]",
                    )
                continue

            total_rows, n_cols = self._get_total_rows(folder)
            linhas_fmt  = f"{total_rows:,}".replace(",", ".") if total_rows  else "N/A"
            colunas_fmt = str(n_cols)                          if n_cols      else "N/A"
            csv_files   = list(folder.glob("*.csv"))
            status_fmt  = "[bold green]✔ Coletado[/]" if csv_files else "[bold red]✖ Ausente[/]"

            tbl.add_row(
                f"{i:02d}", config.get("origem", "—"), config["name"],
                config.get("periodo", "2021–2026"), config.get("freq", "—"),
                colunas_fmt, linhas_fmt, status_fmt,
            )

        console.print(tbl)

    # --------------------------------------------------------------------------
    # 📅 TABELA 2 — STATUS DE ATUALIZAÇÃO
    # --------------------------------------------------------------------------
    def _build_status_table(self) -> None:
        console.print()
        console.print(Panel.fit(
            "[bold white]📅 TABELA 2 — STATUS DE ATUALIZAÇÃO[/]",
            style="bold blue",
        ))

        ref_str = DATA_REF.strftime("%d/%m/%Y %H:%M")
        rows: List[dict] = []

        for config in DATASETS_CONFIG:
            folder    = self.output_dir / config["folder"]
            last_date = self._get_last_date(config["name"], folder)
            label, _  = _classify_status(last_date)
            ultimo_fmt = last_date.strftime("%d/%m/%Y") if last_date is not None else "N/A"
            rows.append({
                "nome":    config["name"],
                "ultimo":  ultimo_fmt,
                "last_ts": last_date if last_date is not None else None,
                "label":   label,
            })

        rows.sort(
            key=lambda r: r["last_ts"] if r["last_ts"] is not None else pd.Timestamp.min,
            reverse=True,
        )

        tbl = Table(box=box.ROUNDED, show_lines=False, header_style="bold cyan")
        tbl.add_column("N°",             style="dim",        width=4,  justify="right")
        tbl.add_column("Dataset",        style="bold white", width=28)
        tbl.add_column("Último Período", width=16,           justify="center")
        tbl.add_column("Data de Ref.",   width=18,           justify="center")
        tbl.add_column("Status",         width=16,           justify="center")

        for i, row in enumerate(rows, start=1):
            label = row["label"]
            if "Atualizado" in label:
                status_fmt = f"[bold green]{label}[/]"
            elif "Atenção" in label:
                status_fmt = f"[bold yellow]{label}[/]"
            elif "Defasado" in label:
                status_fmt = f"[bold red]{label}[/]"
            else:
                status_fmt = f"[dim]{label}[/]"

            tbl.add_row(f"{i:02d}", row["nome"], row["ultimo"], ref_str, status_fmt)

        console.print(tbl)

        total       = len(rows)
        atualizados = sum(1 for r in rows if "Atualizado" in r["label"])
        atencao     = sum(1 for r in rows if "Atenção"    in r["label"])
        defasados   = sum(1 for r in rows if "Defasado"   in r["label"])
        sem_dado    = sum(1 for r in rows if "Sem dado"   in r["label"])

        console.print()
        console.print(Panel(
            f"[bold green]✔ Atualizados: {atualizados}[/]   "
            f"[bold yellow]⚠ Atenção: {atencao}[/]   "
            f"[bold red]✖ Defasados: {defasados}[/]   "
            f"[dim]— Sem dado: {sem_dado}[/]   "
            f"[white]Total: {total}[/]",
            title="Resumo de Status",
            style="bold white",
        ))

    # --------------------------------------------------------------------------
    # 🚀 ORQUESTRADOR PRINCIPAL
    # --------------------------------------------------------------------------
    def run(self) -> None:
        console.clear()
        console.print(Panel.fit(
            "[bold white]🚀 ENERGY DATA COLLECTOR - VERSÃO 10.0[/]",
            style="header",
        ))

        with TimeTracker() as t_total:
            for config in DATASETS_CONFIG:
                t = config["type"]
                if t in ["simple_dict", "simple_template"]:
                    self.process_generic(config)
                elif t == "monthly":
                    self.process_monthly(config)
                elif t == "custom":
                    method = f"process_custom_{config['key'].lower()}"
                    if hasattr(self, method):
                        getattr(self, method)(self.output_dir / config["folder"])

        t_exec = Table(
            title=f"📊 Relatório de Execução  [{t_total.formatted}]",
            box=box.SIMPLE_HEAVY,
        )
        t_exec.add_column("Dataset",   width=26)
        t_exec.add_column("Ano",       width=7)
        t_exec.add_column("Status",    width=12)
        t_exec.add_column("Validação (Lins | MB | Saúde)", width=40)

        status_visible = {"BAIXADO", "ERRO", "EXTRAÍDO", "ATUALIZADO", "PULADO"}
        for l in self.logs:
            if l[2] in status_visible:
                style = (
                    "green"   if l[2] in ["BAIXADO", "EXTRAÍDO", "ATUALIZADO"]
                    else "red"    if l[2] == "ERRO"
                    else "yellow" if l[2] == "PULADO"
                    else "white"
                )
                t_exec.add_row(l[0], l[1], f"[{style}]{l[2]}[/]", l[3])
        console.print(t_exec)

        self.auditar_downloads()
        self.relatorio_datas_finais()

        console.print()
        console.print(Rule("[bold blue]AUDITORIA FINAL DE DATASETS[/]", style="blue"))
        self._build_catalog_table()
        self._build_status_table()

        REPORT_PATH.mkdir(parents=True, exist_ok=True)
        fname = REPORT_PATH / f"Relatorio_Coleta_{DATA_REF.strftime('%Y%m%d_%H%M')}.pdf"
        try:
            doc      = SimpleDocTemplate(str(fname), pagesize=A4)
            elements = [
                Paragraph("Relatório de Coleta — Energy Data v10.0", getSampleStyleSheet()["Title"]),
                Spacer(1, 12),
            ]
            pdf_table = PDFTable(
                [["DATASET", "ANO", "STATUS", "VALIDAÇÃO"]]
                + [[str(x) for x in row[:4]] for row in self.logs]
            )
            pdf_table.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.5, colors.grey)]))
            elements.append(pdf_table)
            doc.build(elements)
            console.print(f"\n📄 [bold green]PDF Gerado:[/] {fname}")
        except Exception as e:
            console.print(f"[yellow]⚠ PDF não gerado: {e}[/]")


# ==============================================================================
# ▶️ ENTRY POINT
# ==============================================================================
if __name__ == "__main__":
    SecureDataCollector(RAW_DATA_PATH).run()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║        TABELAS DE AUDITORIA — CATÁLOGO E STATUS DE DATASETS                ║
# ║        TCC PLD Project — Yago                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# COMO USAR NO GOOGLE COLAB:
#   Célula 1: from google.colab import drive; drive.mount('/content/drive')
#   Célula 2: !pip install rich -q
#   Célula 3: exec(open('tabelas_auditoria_tcc.py').read())
#             OU cole todo o conteúdo deste arquivo em uma célula
#
# O script lê os CSVs já existentes no seu Drive e gera:
#   • Tabela 1 — Catálogo (origem, período, frequência, colunas, linhas, status)
#   • Tabela 2 — Status de atualização (última data capturada vs. hoje)
# ══════════════════════════════════════════════════════════════════════════════

import warnings
import os
from datetime import datetime as dt
from pathlib import Path
from typing import Optional, List, Tuple

import pandas as pd

try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    from rich import box
    from rich.rule import Rule
except ImportError:
    raise SystemExit("Instale o Rich: !pip install rich")

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURAÇÃO — ajuste BASE_PATH se necessário
# ──────────────────────────────────────────────────────────────────────────────
BASE_PATH    = Path("/content/drive/MyDrive/TCC_PLD_Project")
RAW_PATH     = BASE_PATH / "2-DADOS" / "1-DADOS-BRUTOS"
ANO_ATUAL    = dt.now().year
DATA_REF     = dt.now()

console = Console()

# ──────────────────────────────────────────────────────────────────────────────
# CATÁLOGO DE DATASETS
# Cada entrada: chave, nome, pasta, origem, frequência, período, padrão de arquivo
# ──────────────────────────────────────────────────────────────────────────────
# Arquivos exatos a serem lidos por dataset.
# None = lógica especial (INMET: 1 arquivo por subpasta de ano)
# Lista vazia = arquivo único sem padrão de ano
DATASETS = [
    # key,  nome display,                        pasta,                                     origem,  freq,           periodo,      arquivos_exatos
    ("PLD",        "Preço (PLD) CCEE",           "1-CCEE-PLD-HORARIO",                      "CCEE",  "Horária",      "2021–2026",
     ["pld_2021.csv","pld_2022.csv","pld_2023.csv","pld_2024.csv","pld_2025.csv","pld_2026.csv"]),

    ("BALANCO",    "Balanço Energético",          "2-ONS-BALANÇO-DE-ENERGIA-HORARIO",        "ONS",   "Horária",      "2021–2026",
     ["balanco_2021.csv","balanco_2022.csv","balanco_2023.csv","balanco_2024.csv","balanco_2025.csv","balanco_2026.csv"]),

    ("HIDRO",      "Dados Hidrológicos",          "3-ONS-HIDRAULICOS-RESERVATORIOS-HORARIO", "ONS",   "Horária",      "2021–2026",
     ["hidro_2021.csv","hidro_2022.csv","hidro_2023.csv","hidro_2024.csv","hidro_2025.csv","hidro_2026.csv"]),

    ("GERACAO",    "Geração Usinas",              "4-ONS-GERACAO-USINA-HORARIO",             "ONS",   "Horária",      "2021–2026",
     ["geracao_usina_2021.csv","geracao_usina_2022.csv","geracao_usina_2023.csv","geracao_usina_2024.csv","geracao_usina_2025.csv","geracao_usina_2026.csv"]),

    ("CMO",        "Custo Marginal (CMO)",        "5-ONS-CMO-HORARIO",                       "ONS",   "Semi-horária", "2021–2026",
     ["cmo_2021.csv","cmo_2022.csv","cmo_2023.csv","cmo_2024.csv","cmo_2025.csv","cmo_2026.csv"]),

    ("VERTIDA",    "Energia Vertida",             "6-ONS-ENERGIA-VERTIDA-TURBINAVEL",        "ONS",   "Horária",      "2021–2026",
     ["energia_vertida_2021.csv","energia_vertida_2022.csv","energia_vertida_2023.csv","energia_vertida_2024.csv","energia_vertida_2025.csv","energia_vertida_2026.csv"]),

    ("INTERCAMBIO","Intercâmbio Nacional",        "7-ONS-INTERCAMBIO-ENERGIA-NACIONAL",      "ONS",   "Horária",      "2021–2026",
     ["intercambio_ons_2021.csv","intercambio_ons_2022.csv","intercambio_ons_2023.csv","intercambio_ons_2024.csv","intercambio_ons_2025.csv","intercambio_ons_2026.csv"]),

    ("CURVA",      "Curva de Carga",              "8-ONS-CURVA-CARGA",                       "ONS",   "Horária",      "2021–2026",
     ["curva_2021.csv","curva_2022.csv","curva_2023.csv","curva_2024.csv","curva_2025.csv","curva_2026.csv"]),

    ("DISPONIB",   "Disponibilidade Usina",       "9-ONS-DISPONIBILIDADE-USINA",             "ONS",   "Horária",      "2021–2026",
     ["disponibilidade_2021.csv","disponibilidade_2022.csv","disponibilidade_2023.csv","disponibilidade_2024.csv","disponibilidade_2025.csv","disponibilidade_2026.csv"]),

    ("TERMICA",    "Geração Térmica",             "10-ONS-GERACAO-TERMICA-DESPACHO",         "ONS",   "Horária",      "2021–2026",
     ["geracao_termica_2021.csv","geracao_termica_2022.csv","geracao_termica_2023.csv","geracao_termica_2024.csv","geracao_termica_2025.csv","geracao_termica_2026.csv"]),

    ("INMET",      "Meteorologia (INMET)",        "11-INMET-METEOROLOGICOS",                 "INMET", "Horária",      "2021–2026",
     None),  # lógica especial: 1 arquivo por subpasta de ano

    ("EAR",        "Energia Armazenada (EAR)",    "12-ONS-EAR",                              "ONS",   "Diária",       "2021–2026",
     ["ear_2021.csv","ear_2022.csv","ear_2023.csv","ear_2024.csv","ear_2025.csv","ear_2026.csv"]),

    ("ENA",        "Energia Natural (ENA)",       "13-ONS-ENA",                              "ONS",   "Diária",       "2021–2026",
     ["ena_2021.csv","ena_2022.csv","ena_2023.csv","ena_2024.csv","ena_2025.csv","ena_2026.csv"]),

    ("CARGA",      "Carga de Energia",            "14-ONS-CARGA",                            "ONS",   "Diária",       "2021–2026",
     ["carga_2021.csv","carga_2022.csv","carga_2023.csv","carga_2024.csv","carga_2025.csv","carga_2026.csv"]),

    ("CVU",        "Custo Variável (CVU)",        "15-ONS-CVU",                              "ONS",   "Semanal",      "2021–2026",
     ["cvu_2021.csv","cvu_2022.csv","cvu_2023.csv","cvu_2024.csv","cvu_2025.csv","cvu_2026.csv"]),

    ("CARGA_VER",  "Carga Verificada (API)",      "16-ONS-CARGA-VERIFICADA",                 "ONS",   "Horária",      "2021–2026",
     ["carga_verificada_2021_2026.csv"]),  # arquivo único consolidado

    ("INTERCAMBIO2","Intercâmbio ONS",            "17-ONS-INTERCAMBIO",                      "ONS",   "Horária",      "2021–2026",
     ["intercambio_2021.csv","intercambio_2022.csv","intercambio_2023.csv","intercambio_2024.csv","intercambio_2025.csv","intercambio_2026.csv"]),

    ("VOL_ESPERA", "Volume de Espera",            "17-ONS-VOLUME-ESPERA",                    "ONS",   "Diária",       "2021–2026",
     ["volume_espera_2021.csv","volume_espera_2022.csv","volume_espera_2023.csv","volume_espera_2024.csv","volume_espera_2025.csv","volume_espera_2026.csv"]),

    ("FAT_CAP",    "Fator Capacidade",            "18-ONS-FATOR-CAPACIDADE",                 "ONS",   "Diária",       "2021–2026",
     ["fator_capacidade_2021.csv","fator_capacidade_2022.csv","fator_capacidade_2023.csv","fator_capacidade_2024.csv","fator_capacidade_2025.csv","fator_capacidade_2026.csv"]),

    ("DOLAR",      "Cotação Dólar (PTAX)",        "19-BCB-DOLAR",                            "BCB",   "Diária",       "2021–2026",
     ["dolar_ptax_2021_2026.csv"]),  # arquivo único consolidado
]

# Candidatos de coluna de data por dataset key
DATE_HINTS = {
    "PLD":        ["mes_referencia", "din_instante", "data"],
    "BALANCO":    ["din_instante", "data"],
    "HIDRO":      ["din_instante", "data"],
    "GERACAO":    ["din_instante", "data"],
    "CMO":        ["din_instante", "dat_instante", "data"],
    "VERTIDA":    ["din_instante", "data"],
    "INTERCAMBIO":["din_instante", "data"],
    "CURVA":      ["din_instante", "data"],
    "DISPONIB":   ["din_instante", "data"],
    "TERMICA":    ["din_instante", "data"],
    "INMET":      ["data", "hora_utc", "datetime"],
    "EAR":        ["din_instante", "dat_referencia", "data"],
    "ENA":        ["din_instante", "dat_referencia", "data"],
    "CARGA":      ["din_instante", "dat_referencia", "data"],
    "CVU":        ["dat_iniciosemana", "data"],
    "CARGA_VER":  ["din_instante", "datahora", "data"],
    "INTERCAMBIO2":["din_instante", "data"],
    "VOL_ESPERA": ["din_instante", "dat_referencia", "data"],
    "FAT_CAP":    ["din_instante", "dat_referencia", "data"],
    "DOLAR":      ["datahoracotacao", "datahora", "data"],
}

# ──────────────────────────────────────────────────────────────────────────────
# UTILITÁRIOS
# ──────────────────────────────────────────────────────────────────────────────

def detect_inmet_skiprows(path: Path) -> int:
    """
    Arquivos INMET têm metadados nas primeiras linhas antes do cabeçalho real.
    Detecta quantas linhas pular procurando pela linha que começa com 'Data'.
    Caso não encontre, retorna 8 (padrão histórico do INMET).
    """
    for enc in ["latin-1", "utf-8", "utf-8-sig"]:
        try:
            with open(path, encoding=enc, errors="ignore") as f:
                for i, line in enumerate(f):
                    line_strip = line.strip().lower()
                    # Linha de cabeçalho real começa com 'data' seguido de ';' ou 'hora'
                    if line_strip.startswith("data") and ("hora" in line_strip or ";" in line_strip):
                        return i
            break
        except Exception:
            continue
    return 8  # fallback padrão INMET


def read_csv_robust(path: Path, nrows: Optional[int] = None, skiprows: int = 0) -> Optional[pd.DataFrame]:
    """
    Tenta ler CSV testando separadores e encodings.
    Prioriza sep=';' (padrão do pipeline de coleta do TCC).
    Valida que o resultado tem >= 2 colunas E que o número de colunas
    é consistente (maioria das linhas tem o mesmo número de campos).
    """
    best_df: Optional[pd.DataFrame] = None
    best_cols: int = 0

    for sep in [";", ",", "\t"]:
        for enc in ["latin-1", "utf-8", "utf-8-sig", "cp1252"]:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    df = pd.read_csv(
                        path, sep=sep, encoding=enc,
                        on_bad_lines="skip", nrows=nrows,
                        low_memory=False, skiprows=skiprows,
                    )
                n = len(df.columns)
                if n < 2:
                    continue
                # Prefere a leitura com mais colunas (separador mais granular)
                # mas só aceita se for melhor que o atual melhor resultado
                if n > best_cols:
                    best_cols = n
                    best_df = df
                # Sep=';' com encoding correto → retorna imediatamente (padrão do pipeline)
                if sep == ";" and n >= 2:
                    return df
            except Exception:
                continue

    return best_df


def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza nomes de colunas: minúsculas, sem espaços, sem acentos."""
    tbl = str.maketrans("çãâéêíóôúàü", "caaeeiooual")
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .map(lambda c: c.translate(tbl))
    )
    return df


def find_date_column(df: pd.DataFrame, hints: List[str]) -> Optional[str]:
    """
    Detecta coluna de data em ordem de prioridade:
    1. hints explícitos (na ordem fornecida)
    2. varredura por keywords genéricas
    Valida que ao menos 30% dos valores parsam como datetime.
    """
    cols_lower = {c.lower(): c for c in df.columns}

    # Fase 1: hints diretos
    for hint in hints:
        real = cols_lower.get(hint.lower())
        if real:
            parsed = pd.to_datetime(df[real], errors="coerce")
            if parsed.notna().mean() >= 0.30:
                return real

    # Fase 2: keywords genéricas
    keywords = ["instante", "referencia", "datahora", "datetime", "data", "date",
                "hora", "inicio", "cotacao", "semana"]
    for col in df.columns:
        col_l = col.lower()
        if any(k in col_l for k in keywords):
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().mean() >= 0.30:
                return col
    return None


def strip_tz(ts) -> Optional[pd.Timestamp]:
    """Remove timezone de um Timestamp para evitar comparações tz-aware/naive."""
    try:
        ts = pd.Timestamp(ts)
        if ts is pd.NaT:
            return None
        return ts.tz_localize(None) if ts.tzinfo else ts
    except Exception:
        return None


def get_last_date_from_pld(df: pd.DataFrame) -> Optional[pd.Timestamp]:
    """
    PLD CCEE: data construída a partir de mes_referencia + dia + hora.
    Exemplo de mes_referencia: 202101  →  ano=2021, mês=01
    """
    df = normalize_cols(df)
    needed = {"mes_referencia", "dia", "hora"}
    if not needed.issubset(df.columns):
        return None
    try:
        # mes_referencia pode ser 202101 (6 dígitos) ou 2021-01 etc.
        mr = df["mes_referencia"].astype(str).str.replace("-", "").str[:6]
        dia = df["dia"].astype(str).str.zfill(2)
        hora = df["hora"].astype(str).str.zfill(2)
        dt_series = pd.to_datetime(mr + dia + hora, format="%Y%m%d%H", errors="coerce")
        val = dt_series.max()
        return strip_tz(val) if pd.notna(val) else None
    except Exception:
        return None


# ──────────────────────────────────────────────────────────────────────────────
# FUNÇÕES PRINCIPAIS DE LEITURA
# ──────────────────────────────────────────────────────────────────────────────

def get_stats_for_dataset(
    key: str,
    folder: Path,
    arquivos_exatos: Optional[List[str]],
) -> Tuple[int, int, Optional[pd.Timestamp], List[Path]]:
    """
    Retorna (total_linhas, n_colunas, ultima_data, lista_de_arquivos).
    Lê apenas os arquivos listados em arquivos_exatos (nomes exatos, sem glob).
    Para INMET (arquivos_exatos=None), lê 1 arquivo por subpasta de ano.
    """

    # ── INMET: subpastas por ano — lê apenas 1 arquivo por ano (o mais recente) ──
    if key == "INMET":
        # Coleta todas as subpastas de ano (2021, 2022, ...)
        ano_folders = sorted([p for p in folder.iterdir() if p.is_dir()])
        all_sample_files: List[Path] = []

        total_linhas = 0
        n_cols = 0
        max_date: Optional[pd.Timestamp] = None
        hints = DATE_HINTS.get(key, [])

        for ano_folder in ano_folders:
            # Pega todos os CSVs da subpasta e escolhe o mais recente pelo nome
            csvs = sorted(
                list(ano_folder.glob("*.CSV")) + list(ano_folder.glob("*.csv")),
                reverse=True,  # ordem decrescente → o mais recente (alfabeticamente) primeiro
            )
            if not csvs:
                continue

            sample_file = csvs[0]  # arquivo mais recente do ano
            all_sample_files.append(sample_file)

            skiprows = detect_inmet_skiprows(sample_file)
            df = read_csv_robust(sample_file, skiprows=skiprows)
            if df is None or df.empty:
                continue

            # Remove última linha se for rodapé vazio (padrão INMET)
            df = df.dropna(how="all")

            total_linhas += len(df)
            if n_cols == 0:
                n_cols = len(df.columns)

            df_norm = normalize_cols(df.copy())
            date_col = find_date_column(df_norm, hints)
            if date_col:
                ts = strip_tz(pd.to_datetime(df_norm[date_col], errors="coerce").max())
                if ts and (max_date is None or ts > max_date):
                    max_date = ts

        return total_linhas, n_cols, max_date, all_sample_files

    # ── Demais datasets: lê apenas os arquivos exatos listados ──
    csv_files = []
    for nome in (arquivos_exatos or []):
        p = folder / nome
        if p.is_file():
            csv_files.append(p)

    if not csv_files:
        return 0, 0, None, []

    total_linhas = 0
    n_cols = 0
    max_date: Optional[pd.Timestamp] = None
    hints = DATE_HINTS.get(key, [])

    for f in csv_files:
        df = read_csv_robust(f)
        if df is None or df.empty:
            continue

        n_linhas_arquivo = len(df)
        total_linhas += n_linhas_arquivo

        if n_cols == 0:
            n_cols = len(df.columns)

        # Data máxima
        if key == "PLD":
            ts = get_last_date_from_pld(df)
        else:
            df_norm = normalize_cols(df.copy())
            date_col = find_date_column(df_norm, hints)
            ts = None
            if date_col:
                ts = strip_tz(pd.to_datetime(df_norm[date_col], errors="coerce").max())

        if ts and (max_date is None or ts > max_date):
            max_date = ts

    return total_linhas, n_cols, max_date, csv_files


def classify_status(last_date: Optional[pd.Timestamp]) -> Tuple[str, str]:
    """Retorna (label, cor_rich) com base na defasagem."""
    if last_date is None:
        return "Sem dado", "dim"
    try:
        diff = (pd.Timestamp.now() - last_date).days
    except Exception:
        return "Erro", "bold red"
    if diff <= 1:
        return "✔ Atualizado", "bold green"
    elif diff <= 7:
        return "⚠ Atenção", "bold yellow"
    else:
        return "✖ Defasado", "bold red"


# ──────────────────────────────────────────────────────────────────────────────
# GERAÇÃO DAS TABELAS
# ──────────────────────────────────────────────────────────────────────────────

def build_catalog_table() -> List[dict]:
    """Coleta metadados de todos os datasets e retorna lista de dicts."""
    results = []
    total = len(DATASETS)

    console.print()
    with console.status("[cyan]Lendo arquivos CSV…[/]") as status:
        for i, (key, nome, pasta, origem, freq, periodo, arquivos_exatos) in enumerate(DATASETS, 1):
            status.update(f"[cyan]Lendo {nome} ({i}/{total})…[/]")
            folder = RAW_PATH / pasta
            linhas, colunas, ultima_data, arquivos = get_stats_for_dataset(key, folder, arquivos_exatos)
            results.append({
                "key":         key,
                "nome":        nome,
                "pasta":       pasta,
                "origem":      origem,
                "freq":        freq,
                "periodo":     periodo,
                "linhas":      linhas,
                "colunas":     colunas,
                "ultima_data": ultima_data,
                "arquivos":    arquivos,
            })

    return results


def print_table1(data: List[dict]) -> None:
    console.print()
    console.print(Panel.fit(
        "[bold white]📋 TABELA 1 — CATÁLOGO DE DATASETS[/]",
        style="bold blue",
    ))

    tbl = Table(box=box.ROUNDED, show_lines=False, header_style="bold cyan")
    tbl.add_column("N°",         style="dim",        width=4,  justify="right")
    tbl.add_column("Origem",     style="cyan",       width=8)
    tbl.add_column("Dataset",    style="bold white", width=26)
    tbl.add_column("Período",    width=12,           justify="center")
    tbl.add_column("Frequência", width=13,           justify="center")
    tbl.add_column("Colunas",    width=8,            justify="right")
    tbl.add_column("Linhas",     width=15,           justify="right")
    tbl.add_column("Status",     width=14,           justify="center")

    for i, d in enumerate(data, 1):
        linhas_fmt  = f"{d['linhas']:,}".replace(",", ".") if d["linhas"] else "N/A"
        colunas_fmt = str(d["colunas"])                     if d["colunas"] else "N/A"

        if d["key"] == "INMET" and d["arquivos"]:
            status_fmt  = "[dim]ZIP/Pasta[/]"
            linhas_fmt  = f"{d['linhas']:,} lins".replace(",", ".")
            colunas_fmt = str(d["colunas"]) if d["colunas"] else "N/A"
        elif d["arquivos"]:
            status_fmt = "[bold green]✔ Coletado[/]"
        else:
            status_fmt  = "[bold red]✖ Ausente[/]"
            linhas_fmt  = "N/A"
            colunas_fmt = "N/A"

        tbl.add_row(
            f"{i:02d}",
            d["origem"],
            d["nome"],
            d["periodo"],
            d["freq"],
            colunas_fmt,
            linhas_fmt,
            status_fmt,
        )

    console.print(tbl)


def print_table2(data: List[dict]) -> None:
    console.print()
    console.print(Panel.fit(
        "[bold white]📅 TABELA 2 — STATUS DE ATUALIZAÇÃO[/]",
        style="bold blue",
    ))

    ref_str = DATA_REF.strftime("%d/%m/%Y %H:%M")

    # Ordena por última data (mais recente primeiro; sem data no final)
    data_sorted = sorted(
        data,
        key=lambda d: d["ultima_data"] if d["ultima_data"] is not None else pd.Timestamp.min,
        reverse=True,
    )

    tbl = Table(box=box.ROUNDED, show_lines=False, header_style="bold cyan")
    tbl.add_column("N°",             style="dim",        width=4,  justify="right")
    tbl.add_column("Dataset",        style="bold white", width=28)
    tbl.add_column("Último Período", width=16,           justify="center")
    tbl.add_column("Data de Ref.",   width=18,           justify="center")
    tbl.add_column("Status",         width=16,           justify="center")

    contagem = {"Atualizado": 0, "Atenção": 0, "Defasado": 0, "Sem dado": 0}

    for i, d in enumerate(data_sorted, 1):
        ultimo_fmt = (
            d["ultima_data"].strftime("%d/%m/%Y %H:%M")
            if d["ultima_data"] is not None
            else "N/A"
        )
        label, _ = classify_status(d["ultima_data"])

        if "Atualizado" in label:
            status_fmt = f"[bold green]{label}[/]"
            contagem["Atualizado"] += 1
        elif "Atenção" in label:
            status_fmt = f"[bold yellow]{label}[/]"
            contagem["Atenção"] += 1
        elif "Defasado" in label:
            status_fmt = f"[bold red]{label}[/]"
            contagem["Defasado"] += 1
        else:
            status_fmt = f"[dim]{label}[/]"
            contagem["Sem dado"] += 1

        tbl.add_row(f"{i:02d}", d["nome"], ultimo_fmt, ref_str, status_fmt)

    console.print(tbl)
    console.print()
    console.print(Panel(
        f"[bold green]✔ Atualizados: {contagem['Atualizado']}[/]   "
        f"[bold yellow]⚠ Atenção: {contagem['Atenção']}[/]   "
        f"[bold red]✖ Defasados: {contagem['Defasado']}[/]   "
        f"[dim]— Sem dado: {contagem['Sem dado']}[/]   "
        f"[white]Total: {len(data)}[/]",
        title="Resumo de Status",
        style="bold white",
    ))


# ──────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ──────────────────────────────────────────────────────────────────────────────

def main():
    console.print()
    console.print(Panel.fit(
        f"[bold white]📊 AUDITORIA DE DATASETS — TCC PLD Project[/]\n"
        f"[dim]Referência: {DATA_REF.strftime('%d/%m/%Y %H:%M')} | Base: {RAW_PATH}[/]",
        style="bold blue",
    ))

    if not RAW_PATH.exists():
        console.print(f"[bold red]❌ Pasta não encontrada: {RAW_PATH}[/]")
        console.print("[yellow]Monte o Google Drive antes de executar:[/]")
        console.print("  from google.colab import drive; drive.mount('/content/drive')")
        return

    console.print(f"[green]✔ Pasta raiz encontrada:[/] {RAW_PATH}")
    console.print(f"[dim]Subpastas encontradas: {sum(1 for p in RAW_PATH.iterdir() if p.is_dir())}[/]")

    data = build_catalog_table()

    console.print()
    console.print(Rule("[bold blue]RESULTADOS[/]", style="blue"))

    print_table1(data)
    print_table2(data)

    # Sumário rápido de datasets sem arquivos
    ausentes = [d["nome"] for d in data if not d["arquivos"]]
    if ausentes:
        console.print()
        console.print("[bold yellow]⚠ Datasets sem arquivos encontrados:[/]")
        for nome in ausentes:
            console.print(f"  • {nome}")


if __name__ == "__main__":
    main()
else:
    # Permite exec() direto numa célula do Colab
    main()

╭────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 📊 AUDITORIA DE DATASETS — TCC PLD Project                                                         │
│ Referência: 13/06/2026 23:09 | Base: /content/drive/MyDrive/TCC_PLD_Project/2-DADOS/1-DADOS-BRUTOS │
╰────────────────────────────────────────────────────────────────────────────────────────────────────╯

✔ Pasta raiz encontrada: /content/drive/MyDrive/TCC_PLD_Project/2-DADOS/1-DADOS-BRUTOS

Subpastas encontradas: 21

Output()

─────────────────────────────────────────────────── RESULTADOS ────────────────────────────────────────────────────

╭────────────────────────────────────╮
│ 📋 TABELA 1 — CATÁLOGO DE DATASETS │
╰────────────────────────────────────╯

╭─────┬─────────┬───────────────────────────┬─────────────┬─────────────┬─────────┬───────────────┬───────────────╮
│  N° │ Origem  │ Dataset                   │   Período   │ Frequência  │ Colunas │        Linhas │    Status     │
├─────┼─────────┼───────────────────────────┼─────────────┼─────────────┼─────────┼───────────────┼───────────────┤
│  01 │ CCEE    │ Preço (PLD) CCEE          │  2021–2026  │   Horária   │       6 │       188.448 │  ✔ Coletado   │
│  02 │ ONS     │ Balanço Energético        │  2021–2026  │   Horária   │       9 │       235.680 │  ✔ Coletado   │
│  03 │ ONS     │ Dados Hidrológicos        │  2021–2026  │   Horária   │      24 │       845.910 │  ✔ Coletado   │
│  04 │ ONS     │ Geração Usinas            │  2021–2026  │   Horária   │      12 │    30.410.219 │  ✔ Coletado   │
│  05 │ ONS     │ Custo Marginal (CMO)      │  2021–2026  │ Semi-horár… │       4 │       370.176 │  ✔ Coletado   │
│  06 │ ONS     │ Energia Vertida           │  2021–2026  │   Horária   │      18 │     7.029.695 │  ✔ Coletado   │
│  07 │ ONS     │ Intercâmbio Nacional      │  2021–2026  │   Horária   │       6 │       187.680 │  ✔ Coletado   │
│  08 │ ONS     │ Curva de Carga            │  2021–2026  │   Horária   │       4 │       188.256 │  ✔ Coletado   │
│  09 │ ONS     │ Disponibilidade Usina     │  2021–2026  │   Horária   │      13 │    11.404.715 │  ✔ Coletado   │
│  10 │ ONS     │ Geração Térmica           │  2021–2026  │   Horária   │      42 │     5.131.440 │  ✔ Coletado   │
│  11 │ INMET   │ Meteorologia (INMET)      │  2021–2026  │   Horária   │       2 │   45.608 lins │   ZIP/Pasta   │
│  12 │ ONS     │ Energia Armazenada (EAR)  │  2021–2026  │   Diária    │      25 │       134.793 │  ✔ Coletado   │
│  13 │ ONS     │ Energia Natural (ENA)     │  2021–2026  │   Diária    │       7 │         7.856 │  ✔ Coletado   │
│  14 │ ONS     │ Carga de Energia          │  2021–2026  │   Diária    │       4 │         7.856 │  ✔ Coletado   │
│  15 │ ONS     │ Custo Variável (CVU)      │  2021–2026  │   Semanal   │      11 │        28.156 │  ✔ Coletado   │
│  16 │ ONS     │ Carga Verificada (API)    │  2021–2026  │   Horária   │      11 │        20.112 │  ✔ Coletado   │
│  17 │ ONS     │ Intercâmbio ONS           │  2021–2026  │   Horária   │       6 │       188.448 │  ✔ Coletado   │
│  18 │ ONS     │ Volume de Espera          │  2021–2026  │   Diária    │      11 │        59.808 │  ✔ Coletado   │
│  19 │ ONS     │ Fator Capacidade          │  2021–2026  │   Diária    │      21 │     9.105.672 │  ✔ Coletado   │
│  20 │ BCB     │ Cotação Dólar (PTAX)      │  2021–2026  │   Diária    │       3 │         1.348 │  ✔ Coletado   │
╰─────┴─────────┴───────────────────────────┴─────────────┴─────────────┴─────────┴───────────────┴───────────────╯

╭─────────────────────────────────────╮
│ 📅 TABELA 2 — STATUS DE ATUALIZAÇÃO │
╰─────────────────────────────────────╯

╭──────┬──────────────────────────────┬──────────────────┬────────────────────┬──────────────────╮
│   N° │ Dataset                      │  Último Período  │    Data de Ref.    │      Status      │
├──────┼──────────────────────────────┼──────────────────┼────────────────────┼──────────────────┤
│   01 │ Volume de Espera             │ 06/11/2026 00:00 │  13/06/2026 23:09  │   ✔ Atualizado   │
│   02 │ Custo Marginal (CMO)         │ 20/05/2026 23:30 │  13/06/2026 23:09  │    ✖ Defasado    │
│   03 │ Dados Hidrológicos           │ 19/05/2026 19:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   04 │ Balanço Energético           │ 18/05/2026 23:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   05 │ Disponibilidade Usina        │ 18/05/2026 23:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   06 │ Intercâmbio ONS              │ 18/05/2026 23:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   07 │ Energia Armazenada (EAR)     │ 18/05/2026 00:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   08 │ Energia Natural (ENA)        │ 18/05/2026 00:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   09 │ Carga de Energia             │ 18/05/2026 00:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   10 │ Preço (PLD) CCEE             │ 17/05/2026 23:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   11 │ Intercâmbio Nacional         │ 16/05/2026 23:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   12 │ Geração Térmica              │ 16/05/2026 23:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   13 │ Custo Variável (CVU)         │ 16/05/2026 00:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   14 │ Geração Usinas               │ 15/05/2026 23:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   15 │ Curva de Carga               │ 15/05/2026 23:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   16 │ Cotação Dólar (PTAX)         │ 15/05/2026 13:08 │  13/06/2026 23:09  │    ✖ Defasado    │
│   17 │ Energia Vertida              │ 14/05/2026 23:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   18 │ Fator Capacidade             │ 14/05/2026 23:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   19 │ Carga Verificada (API)       │ 16/04/2021 00:00 │  13/06/2026 23:09  │    ✖ Defasado    │
│   20 │ Meteorologia (INMET)         │       N/A        │  13/06/2026 23:09  │     Sem dado     │
╰──────┴──────────────────────────────┴──────────────────┴────────────────────┴──────────────────╯

╭─────────────────────────────────────────────── Resumo de Status ────────────────────────────────────────────────╮
│ ✔ Atualizados: 1   ⚠ Atenção: 0   ✖ Defasados: 18   — Sem dado: 1   Total: 20                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯